In [35]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

In [ ]:
data_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB4/2d-synthetic.csv'
df_train = pd.read_csv(data_path)

data = df_train[['x0', 'x1']]
label = df_train['label']

x_train, x_val, y_train, y_val = train_test_split(data, label, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
x_train.columns

In [ ]:
x_train.describe()

- all num columns, continuous values
- standardize columns

In [ ]:
x_train.isna().sum()

- no nan values
- I can't think of nan-like values for numerical columns / coordinates --> all clean

In [ ]:
label.value_counts()

- it's a 2D space --> plot points positions

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x=x_train['x0'], y=x_train['x1'], hue=label.values[:400])
plt.show()

- check frequencies

In [ ]:
plt.figure(figsize=(8,6))
for plt_idx, col in enumerate(x_train):
    plt.subplot(1,2,plt_idx+1)
    sns.histplot(x=x_train[col], bins='auto', kde=True)
    plt.title(f'frequency {col}')
plt.tight_layout()
plt.show()

- nothing weird
- scale data --> DON'T IF YOU USE THE RANDOM FOREST 

In [ ]:
full_pipeline = Pipeline(steps=[
    (
        'classifier',
        RandomForestClassifier(n_jobs=-1, random_state=42)
    )
])

param_grid = {
    'classifier__n_estimators':[50,100,150],
    'classifier__max_depth':[None],
    'classifier__min_samples_split':[2,5],
    'classifier__min_samples_leaf':[1,2,5],
}

grid = GridSearchCV(estimator=full_pipeline, param_grid=param_grid, cv=4, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

print(grid.best_params_)
best_model = grid.best_estimator_

y_pred_train = best_model.predict(x_train)
print(classification_report(y_train, y_pred_train))

y_pred = best_model.predict(x_val)
print(classification_report(y_val, y_pred))


In [ ]:
full_pipeline = Pipeline(steps=[
    (
        'scaler',
        StandardScaler()
    )
    ,
    (
        'classifier',
        LinearSVC(random_state=42)
    )
])

param_grid = {
    'classifier__penalty':['l1', 'l2'],
    'classifier__loss':['squared_hinge', 'hinge'],
    'classifier__C':[0.5,1,2]
    }

grid = GridSearchCV(estimator=full_pipeline, param_grid=param_grid, cv=4, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

print(grid.best_params_)
best_model = grid.best_estimator_

y_pred_train = best_model.predict(x_train)
print(classification_report(y_train, y_pred_train))

y_pred = best_model.predict(x_val)
print(classification_report(y_val, y_pred))

In [ ]:
full_pipeline = Pipeline(steps=[
    (
        'scaler',
        StandardScaler()
    )
    ,
    (
        'classifier',
        KNeighborsClassifier(n_jobs=-1)
    )
])

param_grid = {
    'classifier__n_neighbors':[2, 5, 10],
    'classifier__weights':['uniform', 'distance'],
    }

grid = GridSearchCV(estimator=full_pipeline, param_grid=param_grid, cv=4, n_jobs=-1, verbose=1)
grid.fit(x_train, y_train)

print(grid.best_params_)
best_model = grid.best_estimator_

y_pred_train = best_model.predict(x_train)
print(classification_report(y_train, y_pred_train))

y_pred = best_model.predict(x_val)
print(classification_report(y_val, y_pred))

# MAIN

In [ ]:
def get_data():
    data_path = '/Users/filippomontecchi/Desktop/Data Science & Machine Learning Lab/Lab/Dataset/LAB4/2d-synthetic.csv'
    df_train = pd.read_csv(data_path)

    x_train = df_train[['x0', 'x1']]
    y_train = df_train['label']

    return x_train, y_train

def full_pipeline(x_train, y_train):
    full_pipeline = Pipeline(steps=[
        (
            'scaler',
            StandardScaler()
        )
        ,
        (
            'classifier',
            LinearSVC(random_state=42)
        )
    ])

    param_grid = {
        'classifier__penalty':['l1', 'l2'],
        'classifier__loss':['squared_hinge', 'hinge'],
        'classifier__C':[0.5,1,2]
        }

    grid = GridSearchCV(estimator=full_pipeline, param_grid=param_grid, cv=4, n_jobs=-1, verbose=1)
    grid.fit(x_train, y_train)

    print(grid.best_params_)
    best_model = grid.best_estimator_

    y_pred_train = best_model.predict(x_train)
    print(classification_report(y_train, y_pred_train))

if __name__ == '__main__':
    x_train, y_train = get_data()
    full_pipeline(x_train, y_train)
